# Point-in-Time 基本面因子研究

这个 Notebook 按实验执行顺序展示数据生成、PIT 对齐、因子构建和评价。默认数据为固定随机种子的模拟样本，只用于复现研究流程。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root / 'src'))

from fundamental_alpha.synthetic import generate_demo_data
from fundamental_alpha.point_in_time import align_fundamentals_asof
from fundamental_alpha.diagnostics import build_data_quality_report
from fundamental_alpha.factors import build_factors
from fundamental_alpha.evaluation import (
    monthly_rank_ic, ic_summary, quantile_returns,
    performance_summary, decay_analysis, style_exposure,
)

## 1. 生成输入并检查字段

财务表保留报告期和公告日；行情表按月保存信号日截面与下一持有期收益。

In [ ]:
fundamentals, market = generate_demo_data(seed=42)
display(fundamentals.head())
display(market.head())
print(f'fundamentals: {fundamentals.shape}; market: {market.shape}')

## 2. 按公告日进行 PIT 对齐

每个信号日只使用当时已经公告的最近一期财报。

In [ ]:
aligned = align_fundamentals_asof(market, fundamentals)
quality_report = build_data_quality_report(fundamentals, market, aligned)
quality_report

## 3. 构建并检查复合因子

原始指标经过 MAD 去极值、截面标准化以及行业和对数市值中性化。

In [ ]:
panel = build_factors(aligned)
factor = 'factor_composite'
cross_section_check = panel.groupby('date')[factor].agg(['count', 'mean', 'std'])
display(cross_section_check.head())

## 4. IC、分层、衰减和样本外

2022-01-01 之后作为固定样本外区间。

In [ ]:
ic = monthly_rank_ic(panel, factor)
quantiles = quantile_returns(panel, factor)
split = '2022-01-01'

result = {
    'full_sample_ic': ic_summary(ic),
    'out_of_sample_ic': ic_summary(monthly_rank_ic(panel[panel.date >= split], factor)),
    'long_short': performance_summary(quantiles.long_short),
    'style_exposure': style_exposure(panel, factor),
}
result

In [ ]:
display(quantiles[['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'long_short']].mean().to_frame('monthly_mean'))
display(decay_analysis(panel, factor))

## 5. 结果解释

判断顺序是：先检查 PIT 覆盖与异常记录，再看 IC 方向、分层单调性和样本外稳定性，最后观察换手、衰减与风格暴露。模拟数据的作用是验证管线，不用于证明真实 Alpha。完整实验可通过 `fundamental-alpha demo --output outputs/demo` 重新运行。